In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

# In this notebook we check the reparametrization step. The goal is to get $|\phi_{i+1}^n-\phi_{i}^n|$ constant

In [ ]:
@inline lerp(a,b,t) = (1-t)*a + t*b

"""
Reparameterize a discrete curve φ (d×N) to have uniform arc-length spacing.

Inputs
- φ::AbstractMatrix{T}: size (d, N), columns are points φ[:,i] ∈ R^d
- payload::AbstractArray: either
    * vector length N, or
    * matrix size (k, N) (any "data attached to points")
- Nout::Int: number of output points (default = N)

Options
- epslen: threshold to ignore zero-length segments

Returns
- φr: (d, Nout) reparameterized points
- payloadr: payload interpolated to the new parameterization (same "shape", but Nout points)
"""
function reparametrization_arc_length(
    φ::AbstractMatrix{T},
    payload::AbstractArray,
    Nout::Int = size(φ, 2);
    epslen::T = T(1e-14),
) where {T<:Real}

    d, N = size(φ)
    @assert N ≥ 2 "Need at least 2 points"

    # --- cumulative arc-length s[i] ---
    s = zeros(T, N)
    @inbounds for i in 2:N
        # Euclidean distance between consecutive points
        di = zero(T)
        @inbounds for k in 1:d
            δ = φ[k,i] - φ[k,i-1]
            di += δ*δ
        end
        di = sqrt(di)
        s[i] = s[i-1] + (di > epslen ? di : zero(T))
    end

    L = s[end]
    if L ≤ epslen
        # Degenerate curve: all points identical (or nearly). Return copies.
        φr = repeat(φ[:, 1:1], 1, Nout)
        payloadr = _payload_repeat_first(payload, Nout)
        return φr, payloadr
    end

    # Uniform target arc-length positions
    sr = range(zero(T), L, length = Nout)

    # Output containers
    φr = similar(φ, d, Nout)
    payloadr = _payload_alloc_like(payload, Nout)

    # --- helper: linear interpolation at target s ---
    # For each target sr[j], find segment i-1 -> i s.t. s[i-1] <= sr <= s[i]
    @inbounds for j in 1:Nout
        sj = sr[j]
        i = searchsortedfirst(s, sj)  # i in 1..N
        if i <= 1
            # clamp to first point
            @inbounds for k in 1:d
                φr[k,j] = φ[k,1]
            end
            _payload_set!(payloadr, payload, j, 1, one(T), zero(T))
        elseif i > N
            # clamp to last point
            @inbounds for k in 1:d
                φr[k,j] = φ[k,N]
            end
            _payload_set!(payloadr, payload, j, N, one(T), zero(T))
        else
            s0, s1 = s[i-1], s[i]
            t = (s1 - s0) ≤ epslen ? zero(T) : (sj - s0) / (s1 - s0)

            @inbounds for k in 1:d
                φr[k,j] = (one(T)-t)*φ[k,i-1] + t*φ[k,i]
            end
            _payload_set!(payloadr, payload, j, i, t, one(T)-t)
        end
    end

    return φr, payloadr
end

# ---- payload helpers (supports vector length N or matrix k×N) ----

# Allocate payload-like container with Nout points
function _payload_alloc_like(payload::AbstractVector{T}, Nout::Int) where {T}
    return similar(payload, T, Nout)
end
function _payload_alloc_like(payload::AbstractMatrix{T}, Nout::Int) where {T}
    k, _ = size(payload)
    return similar(payload, T, k, Nout)
end

# If curve is degenerate, repeat first payload entry
function _payload_repeat_first(payload::AbstractVector{T}, Nout::Int) where {T}
    out = similar(payload, T, Nout)
    out .= payload[1]
    return out
end
function _payload_repeat_first(payload::AbstractMatrix{T}, Nout::Int) where {T}
    k, _ = size(payload)
    out = similar(payload, T, k, Nout)
    @inbounds for j in 1:Nout
        out[:,j] .= payload[:,1]
    end
    return out
end

# Write interpolated payload at output index j.
# Convention: i is the RIGHT endpoint index (segment i-1 -> i).
function _payload_set!(out::AbstractVector{T}, payload::AbstractVector{T},
                      j::Int, i::Int, t::T, w0::T) where {T}
    # out[j] = (1-t)*payload[i-1] + t*payload[i]
    if i == 1
        out[j] = payload[1]
    else
        out[j] = w0*payload[i-1] + t*payload[i]
    end
end
function _payload_set!(out::AbstractMatrix{T}, payload::AbstractMatrix{T},
                      j::Int, i::Int, t::T, w0::T) where {T}
    k, _ = size(payload)
    if i == 1
        @inbounds for r in 1:k
            out[r,j] = payload[r,1]
        end
    else
        @inbounds for r in 1:k
            out[r,j] = w0*payload[r,i-1] + t*payload[r,i]
        end
    end
end

In [ ]:
φ_test = [[0.0,0.0] [1.0,2.0] [2.0,1.0]]
plot(φ_test[1,:], φ_test[2,:])

In [ ]:
d, N = size(φ_test)

# --- cumulative arc-length s[i] ---
s = zeros(T, N)
for i in 2:N
    # Euclidean distance between consecutive points
    di = norm(φ_test[:,i]- φ_test[:,i-1])
    s[i] = s[i-1] + di
end
L = s[end]
# Uniform target arc-length positions
sr = range(0.0, L, length = N)

# Output containers
φr = similar(φ_test, d, N)

for j in 1:N
    sj = sr[j]
    i = searchsortedfirst(s, sj)  # i in 1..N
    if i <= 1
        # clamp to first point
        for k in 1:d
            φr[k,j] = φ_test[k,1]
        end
    elseif i > N
        # clamp to last point
        for k in 1:d
            φr[k,j] = φ_test[k,N]
        end
    else
        s0, s1 = s[i-1], s[i]
        t = (sj - s0) / (s1 - s0)

        for k in 1:d
            φr[k,j] = (one(T)-t)*φ_test[k,i-1] + t*φ_test[k,i]
        end
    end
end
φr

In [ ]:
plot(φ_test[1,:], φ_test[2,:])
plot!(φr[1,:], φr[2,:])

In [ ]:
s = zeros(T, N)
@inbounds for i in 2:N
    # Euclidean distance between consecutive points
    di = zero(T)
    @inbounds for k in 1:d
        δ = φr[k,i] - φr[k,i-1]
        di += δ*δ
    end
    di = sqrt(di)
    s[i] = s[i-1] + (di > epslen ? di : zero(T))
end
plot(s)
hline!([norm(φr[:,2]- φr[:,1])])
hline!([norm(φr[:,3]- φr[:,2]) + norm(φr[:,2]- φr[:,1])])

In [ ]:
d, N = size(φ_test)
φ_test= φr
# --- cumulative arc-length s[i] ---
s = zeros(T, N)
for i in 2:N
    # Euclidean distance between consecutive points
    di = norm(φ_test[:,i]- φ_test[:,i-1])
    s[i] = s[i-1] + di
end
L = s[end]
# Uniform target arc-length positions
sr = range(0.0, L, length = N)

# Output containers
φr = similar(φ_test, d, N)

for j in 1:N
    sj = sr[j]
    i = searchsortedfirst(s, sj)  # i in 1..N
    if i <= 1
        # clamp to first point
        for k in 1:d
            φr[k,j] = φ_test[k,1]
        end
    elseif i > N
        # clamp to last point
        for k in 1:d
            φr[k,j] = φ_test[k,N]
        end
    else
        s0, s1 = s[i-1], s[i]
        t = (sj - s0) / (s1 - s0)

        for k in 1:d
            φr[k,j] = (one(T)-t)*φ_test[k,i-1] + t*φ_test[k,i]
        end
    end
end
φr

In [ ]:
plot(φ_test[1,:], φ_test[2,:])
plot!(φr[1,:], φr[2,:])

In [ ]:
norm(φr[:,3]- φr[:,2])

In [ ]:
norm(φr[:,2]- φr[:,1])

In [ ]:
φ_test[:,1]

In [ ]:
using LinearAlgebra

φ_test = [0.0 0.0;
          1.0 2.0;
          2.0 1.0]

p1 = φ_test[1, :]
p2_old = φ_test[2, :]
p3 = φ_test[3, :]

v = p2_old - p1
w = p3 - p1

den = 2 * dot(w, v)

if iszero(den)
    error("No solution: the constraint line is incompatible with equal distances.")
end

t = dot(w, w) / den

if !(0.0 <= t <= 1.0)
    error("No solution on the segment [p1, p2_old], because t = $t is outside [0,1].")
end

p2_new = p1 + t * v

φr = copy(φ_test)
φr[2, :] = p2_new

d1 = norm(φr[2, :] - φr[1, :])
d2 = norm(φr[3, :] - φr[2, :])

println("t = ", t)
println("new middle point = ", p2_new)
println("updated matrix = ")
println(φr)
println("norm(p2-p1) = ", d1)
println("norm(p3-p2) = ", d2)
plot(φ_test[:,1], φ_test[:,2])
plot!(φr[:,1], φr[:,2])

In [ ]:
using LinearAlgebra

φ_test = [0.0 0.0;
          1.0 2.0;
          2.0 1.0;
          3.0 2.0]

p1 = φ_test[1,:]
p2_old = φ_test[2,:]
p3_old = φ_test[3,:]
p4 = φ_test[4,:]

v2 = p2_old - p1
v3 = p3_old - p4

# function whose root we want
function f(x)
    t2, t3 = x

    p2 = p1 + t2*v2
    p3 = p4 + t3*v3

    d12 = norm(p2-p1)
    d23 = norm(p3-p2)
    d34 = norm(p4-p3)

    return [d12-d23,
            d23-d34]
end

using NLsolve

sol = nlsolve(f, [0.5,0.5])

t2, t3 = sol.zero

p2 = p1 + t2*v2
p3 = p4 + t3*v3

φr = copy(φ_test)
φr[2,:] = p2
φr[3,:] = p3

println("rescaled points:")
println(φr)

println("distances:")
println(norm(φr[2,:]-φr[1,:]))
println(norm(φr[3,:]-φr[2,:]))
println(norm(φr[4,:]-φr[3,:]))
plot(φ_test[:,1], φ_test[:,2])
plot!(φr[:,1], φr[:,2])

In [ ]:
using LinearAlgebra
using NLsolve

# Evaluate point on polyline at arc-length parameter s
function point_on_polyline(φ::AbstractMatrix, svals::AbstractVector, seglen::AbstractVector, s::Real)
    N, d = size(φ)

    if s <= 0
        return Vector{Float64}(φ[1, :])
    elseif s >= svals[end]
        return Vector{Float64}(φ[end, :])
    end

    j = searchsortedlast(svals, s)
    j = min(j, N - 1)

    if iszero(seglen[j])
        return Vector{Float64}(φ[j, :])
    end

    t = (s - svals[j]) / seglen[j]
    return (1 - t) .* φ[j, :] .+ t .* φ[j + 1, :]
end

function equal_euclidean_points_on_curve(φ::AbstractMatrix)
    N, d = size(φ)
    N >= 2 || error("Need at least 2 points")

    seglen = [norm(φ[i+1, :] - φ[i, :]) for i in 1:N-1]
    L = sum(seglen)

    # cumulative arc length
    svals = zeros(Float64, N)
    for i in 2:N
        svals[i] = svals[i-1] + seglen[i-1]
    end

    # initial guess: uniform in arc length
    x0 = collect(range(0, L, length=N)[2:end-1])

    function F!(Fvec, x)
        s = vcat(0.0, x, L)
        pts = [point_on_polyline(φ, svals, seglen, si) for si in s]

        dists = [norm(pts[i+1] - pts[i]) for i in 1:N-1]
        for i in 1:N-2
            Fvec[i] = dists[i] - dists[i+1]
        end
    end

    sol = nlsolve(F!, x0)
    x = sol.zero
    s = vcat(0.0, x, L)

    φnew = zeros(Float64, N, d)
    for i in 1:N
        φnew[i, :] .= point_on_polyline(φ, svals, seglen, s[i])
    end

    return φnew, s
end

φ_test = [0.0 0.0;
          0.1 0.0;
          2.0 1.0;
          3.0 2.0]

φr, s = equal_euclidean_points_on_curve(φ_test)

println("Updated points:")
println(φr)

println("\nArc-length parameters:")
println(s)

println("\nEuclidean distances:")
for i in 1:size(φr,1)-1
    println(norm(φr[i+1,:] - φr[i,:]))
end
plot(φ_test[:,1], φ_test[:,2])
plot!(φr[:,1], φr[:,2])

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_a^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2
E_J = 37.12*2pi

#Environment
κ_b=1/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.2
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
ε_d_p = ε_d*φ_b/ω_b
κ_b_p = κ_b / ω_b

ℵ = β_a_p*β_b_p/(4*κ_b_p)
ℶ = β_a_p*ε_d_p/(2*κ_b_p)
function equation_2modes!(du,u,params,t)
    β_a_p,β_b_p,ε_d_p,κ_b_p = params

    x_a = u[1]
    p_a = u[2]
    x_b = u[3]
    p_b = u[4]
        
    du[1] = β_a_p*(x_a*p_b-p_a*x_b)/sqrt(2)
    du[2] = -β_a_p*(x_a*x_b+p_a*p_b)/sqrt(2)
    du[3] = 2*(β_b_p*x_a*p_a/sqrt(2)-κ_b_p*x_b)
    du[4] = 2*(-β_b_p*(x_a^2-p_a^2)/2/sqrt(2)-ε_d_p/sqrt(2)-κ_b_p*p_b)
    return nothing
end
t_min = 0.0
t_max = 1e1/κ_b_p 
tspan = (t_min, t_max)

a_0 = 0.0-0.01im 
b_0 = -1im*ε_d_p/2/κ_b_p#0

x_a0, p_a0, x_b0, p_b0 = real(a_0)*sqrt(2), imag(a_0)*sqrt(2), real(b_0)*sqrt(2), imag(b_0)*sqrt(2)
u_0 = [x_a0, p_a0, x_b0, p_b0]
params = (β_a_p,β_b_p,ε_d_p,κ_b_p)
prob = ODEProblem(equation_2modes!, u_0, tspan, params, saveat=range(t_min,t_max, length=500))
sol = solve(prob, Tsit5())
t = sol.t
x_a = [u[1] for u in sol.u]
p_a = [u[2] for u in sol.u]
x_b = [u[3] for u in sol.u]
p_b = [u[4] for u in sol.u];
x_all = -.2:0.01:.2
p_all = -.35:0.01:.35

U = [ℵ*(x^2 + p^2)^2/4 + ℶ*(x^2 - p^2)/2 for p in p_all, x in x_all]

heatmap(x_all, p_all, U)
plot!(x_a,p_a, label="Trajectory")

In [ ]:
ϕ_diss = cat(x_a[end:-1:1], p_a[end:-1:1], dims = 2)
φr, s = equal_euclidean_points_on_curve(ϕ_diss)

println("Updated points:")
println(φr)

println("\nArc-length parameters:")
println(s)

println("\nEuclidean distances:")
for i in 1:size(φr,1)-1
    println(norm(φr[i+1,:] - φr[i,:]))
end
scatter(φr[:,2])
scatter!(ϕ_diss[:,2])

In [ ]:
size(φr, 1)

In [ ]:
gaps = [norm(φr[i+1,:] - φr[i,:]) for i in 1:(size(φr, 1)-1)]
plot(gaps)